# Session 2.2 — Context Management & Chat History

## What We're Building Today

A working Streamlit chat application that:
- Rewrites follow-up queries for better retrieval
- Assembles retrieved chunks into coherent reading order (grouped by document, sorted by index, gap-marked)
- Manages conversation history via truncation to prevent context overflow

## Learning Objectives

1. **Explain** why follow-up questions fail in stateless retrieval and how query rewriting solves the problem
2. **Implement** `contextualize_query()` to rewrite follow-up questions using conversation history and an LLM call
3. **Assemble** retrieved chunks into coherent reading order by grouping by source document, sorting by chunk index, and marking gaps
4. **Manage** chat history within context window limits using a message-count sliding-window truncation strategy

## Session Theme

> "Retrieval is stateless. Your user is not."

In [ ]:
# Path setup — run this cell FIRST
# Notebooks run from their subdirectory but pipeline modules are at the project root
import sys
from pathlib import Path

_PROJECT_ROOT = Path.cwd().parent.parent
assert (_PROJECT_ROOT / "pipeline").exists(), (
    f"pipeline/ not found at {_PROJECT_ROOT}. "
    "Make sure you opened the notebook from the AI-3/notebooks/2_2/ directory."
)
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

# Load environment variables from project root
from dotenv import load_dotenv
load_dotenv(_PROJECT_ROOT / ".env")

print(f"Project root: {_PROJECT_ROOT}")
print(f"Path configured: pipeline imports will work")

In [ ]:
# Import verification
from pipeline.retrieval.naive import naive_retrieve
from pipeline.generation.generate import call_claude
from pipeline.context.assembler import assemble_context, contextualize_query
from pipeline.context.manager import manage_history

print("All pipeline imports successful!")
print("  - pipeline.retrieval.naive")
print("  - pipeline.generation.generate")
print("  - pipeline.context.assembler")
print("  - pipeline.context.manager")

In [ ]:
# ChromaDB verification
from pipeline.ingestion.store import get_collection

collection = get_collection()
count = collection.count()
print(f"ChromaDB collection: {collection.name}")
print(f"Documents loaded: {count} chunks")
assert count > 0, "No chunks found! Did you run ingestion?"

In [ ]:
# Claude API test
response = call_claude("Say hello in 3 words")
print(f"Claude says: {response}")

In [ ]:
# Verify upgraded rag.py works
from app.rag import get_response, ChatResponse

test_response = get_response("What does Northbrook do?", messages=[])
print(f"Answer: {test_response.answer[:100]}...")
print(f"Sources: {len(test_response.sources)} chunks retrieved")
print(f"Type: {type(test_response).__name__}")

### Setup Checklist

- [ ] All pipeline imports successful
- [ ] ChromaDB collection has chunks loaded
- [ ] Claude API responds
- [ ] Upgraded `app/rag.py` works (returns ChatResponse with answer and sources)

If anything above failed, run `git pull` to get the latest `app/rag.py` upgrade.

---

## The Follow-up Problem

Retrieval is **stateless** — it embeds a single query string. It does not see conversation history.

When a user asks a follow-up question like "How many days do I get?" after discussing vacation policy, the retrieval system embeds exactly those six words. It has no idea the user is talking about vacation days.

```
WHAT THE USER MEANT:
  "How many vacation days does a Northbrook employee receive?"

WHAT RETRIEVAL SAW:
  "How many days do I get?"
```

Let's see this failure in action.

In [ ]:
# Demo: Stateless retrieval failure

# Question 1: Standalone — retrieves well
print("=" * 60)
print("Question 1: 'What is the vacation policy at Northbrook?'")
print("=" * 60)
results_1 = naive_retrieve("What is the vacation policy at Northbrook?", top_k=3)
for r in results_1:
    print(f"  [{r['score']:.3f}] {r['metadata']['source']}: {r['text'][:60]}...")

print()

# Question 2: Follow-up — retrieves poorly
print("=" * 60)
print("Question 2 (follow-up): 'How many days do I get?'")
print("=" * 60)
results_2 = naive_retrieve("How many days do I get?", top_k=3)
for r in results_2:
    print(f"  [{r['score']:.3f}] {r['metadata']['source']}: {r['text'][:60]}...")

print()
print("Notice the score drop on the follow-up question.")

### The Fix: Rewrite the Query Before Retrieval

Claude can read the conversation history and understand what the user means. So before we send the query to retrieval, we ask Claude to rewrite it.

"How many days do I get?" becomes "How many vacation days does a Northbrook employee receive?" — and that retrieves perfectly.

```
RETRIEVAL SCORES:
  Original follow-up:  0.43, 0.41, 0.38  (bad)
  Rewritten query:     0.84, 0.81, 0.78  (good)
```

One extra Claude call before retrieval. The cost: ~0.3 seconds and a few hundred tokens. The benefit: follow-up questions actually work.

---

## Query Rewriting: Build `contextualize_query()`

Open `pipeline/context/assembler.py`. Find `contextualize_query()`. It currently has a **passthrough default** — returns the user message unchanged. Replace it with an LLM call.

**The logic:**
1. If there is no history (first message), return the query unchanged
2. Format the recent history (last 3-4 exchanges) as a string
3. Call Claude with a rewriting prompt
4. Return the rewritten query

**The rewriting prompt template:**

```
Given the conversation history below, rewrite the user's latest
message as a standalone question suitable for a search engine.
If the message is already standalone, return it unchanged.

Return ONLY the rewritten question — no explanation, no preamble.

History:
{recent_history}

Latest message: {user_message}

Rewritten question:
```

**Tips:**
- Limit history to the last 6-8 messages (3-4 exchanges)
- Use `temperature=0.0` for deterministic output
- Use `call_claude()` from `pipeline.generation.generate`

In [ ]:
# YELLOW scaffold — implement contextualize_query()
# Implement this in pipeline/context/assembler.py

def contextualize_query(history: list[dict], user_message: str) -> str:
    """Rewrite a follow-up question so it stands alone for retrieval.

    When a user asks "How many days do I get?" after discussing vacation
    policy, retrieval needs the full question: "How many vacation days
    does a Northbrook employee receive?"

    If the message is already self-contained, return it unchanged.

    Args:
        history: Previous conversation messages (list of role/content dicts).
        user_message: The user's latest message.

    Returns:
        A rewritten query that stands alone for retrieval,
        OR the original user_message if no rewrite is needed.
    """
    # Step 1: Check for empty history — return early if no history

    # Step 2: Format recent history (last 6 messages) as a string
    #         Loop through messages, format as "{role}: {content}"

    # Step 3: Build the rewriting prompt using the template above

    # Step 4: Call Claude with temperature=0.0 and max_tokens=256
    #         Use call_claude(prompt=..., system_prompt=..., temperature=0.0, max_tokens=256)

    # Step 5: Return the rewritten query (stripped of whitespace)
    pass

> **Important:** Implement this function in the `.py` file (`pipeline/context/assembler.py`), not in this notebook cell. The cell above shows the structure — your implementation must be in the `.py` file for the app to use it. After saving, restart the kernel and run the test cell below.

In [ ]:
# Test your contextualize_query() implementation
from pipeline.context.assembler import contextualize_query

# Simulate a conversation about vacation policy
fake_history = [
    {"role": "user", "content": "What is the vacation policy at Northbrook?"},
    {"role": "assistant", "content": "Northbrook provides employees with paid time off based on tenure..."}
]

# This follow-up should be rewritten to include "vacation"
follow_up = "How many days do I get?"
rewritten = contextualize_query(fake_history, follow_up)

print(f"Original:  {follow_up}")
print(f"Rewritten: {rewritten}")
print()

# Test: first message (no history) should return unchanged
standalone = contextualize_query([], "What is the vacation policy?")
print(f"Standalone (no history): {standalone}")
print()

# Verify the rewritten query retrieves better
print("Retrieval with rewritten query:")
results = naive_retrieve(rewritten, top_k=3)
for r in results:
    print(f"  [{r['score']:.3f}] {r['text'][:60]}...")

### What Did You Notice?

- Does the rewritten query include the topic from the conversation ("vacation")?
- Are the retrieval scores higher with the rewritten query?
- What happens when the message is already standalone?
- How fast was the rewriting call? (Should be <1 second)

---

## Context Assembly: Build `assemble_context()`

Right now, chunks go into the system prompt sorted by cosine similarity — highest score first. This is like ripping pages out of two different books, shuffling them by relevance, and handing someone the stack.

`assemble_context()` reorganizes chunks into **reading order** — same chunks, zero additional API cost, better answers.

```
BEFORE (similarity order — what ChromaDB returns):
  q3-financial.md chunk 5 (0.89) -> board-meeting.md chunk 3 (0.85) ->
  q3-financial.md chunk 2 (0.81) -> board-meeting.md chunk 1 (0.77) ->
  q3-financial.md chunk 4 (0.72)

AFTER (assemble_context — what Claude reads):

  --- Source: q3-financial.md ---
  [chunk 2] "...this report covers Q3 financial performance..."
    [...content omitted...]
  [chunk 4] "...operating expenses remained within budget..."
  [chunk 5] "...Q3 revenue totaled $4.2M, exceeding the..."

  --- Source: board-meeting.md ---
  [chunk 1] "...the board convened to review quarterly..."
    [...content omitted...]
  [chunk 3] "...projections for Q3 were discussed at..."
```

**Three things happen:**
1. Chunks are **grouped** by source document
2. Within each group, chunks are **sorted** by chunk index (original document order)
3. When there is a gap in the index (non-consecutive), insert a **gap marker**

**The algorithm:**
```
assemble_context(chunks):
  1. Group chunks by source document (metadata['source'])
  2. Sort each group by chunk index (metadata['chunk_index'])
  3. For each group:
     a. Write the document header: "--- Source: {source} ---"
     b. For each chunk: if not consecutive with previous, insert gap marker
     c. Write the chunk text
  4. Join groups and return
```

In [ ]:
# YELLOW scaffold — implement assemble_context()
# Implement this in pipeline/context/assembler.py

from collections import defaultdict


def assemble_context(
    chunks: list[dict], gap_marker: str = "[...content omitted...]"
) -> str:
    """Assemble retrieved chunks into coherent reading order.

    Steps:
      1. Group chunks by source document (metadata['source'])
      2. Sort each group by chunk index (metadata['chunk_index'])
      3. For each group, build a section:
         - Header: \"--- Source: {source} ---\"
         - For each chunk: if chunk_index is not consecutive
           with the previous chunk, insert the gap_marker
         - Then the chunk text
      4. Join all sections with double newlines
      5. Return the assembled context string

    Args:
        chunks: List of chunk dicts with keys: text, metadata (source, chunk_index), score
        gap_marker: String to insert between non-consecutive chunks.

    Returns:
        Assembled context string with document grouping, reading order, and gap markers.
    """
    if not chunks:
        return ""

    # Step 1: Group chunks by source document using defaultdict(list)

    # Step 2: For each group, sort by chunk_index
    #         sorted(group, key=lambda c: c['metadata']['chunk_index'])

    # Step 3: Build sections with headers and gap detection
    #         Track prev_index. If curr_index > prev_index + 1, insert gap_marker

    # Step 4: Join all sections with double newlines

    # Step 5: Return the assembled string
    pass

> **Important:** Implement this function in the `.py` file (`pipeline/context/assembler.py`), not in this notebook cell. The cell above shows the structure — your implementation must be in the `.py` file for the app to use it. After saving, restart the kernel and run the test cell below.

In [ ]:
# Test your assemble_context() implementation
from pipeline.context.assembler import assemble_context

# Retrieve some chunks
chunks = naive_retrieve("What is the vacation policy at Northbrook?", top_k=5)

# Show what ChromaDB returns (similarity order)
print("BEFORE — Similarity order from ChromaDB:")
print("-" * 50)
for i, c in enumerate(chunks):
    source = c['metadata'].get('source', 'Unknown')
    idx = c['metadata'].get('chunk_index', '?')
    print(f"  {i+1}. [{c['score']:.3f}] {source} (chunk {idx}): {c['text'][:50]}...")

print()

# Run through assemble_context
assembled = assemble_context(chunks)
print("AFTER — assemble_context() output:")
print("-" * 50)
print(assembled)

### What Did You Notice?

- Are chunks now grouped by source document?
- Within each group, are they in reading order (sorted by chunk index)?
- Do you see gap markers where chunks are not consecutive?
- Same chunks, same cost — but now Claude reads them in coherent order.

---

## Token Management: Build `manage_history()`

### Context Window Anatomy

Claude Sonnet 4.5 has a 200,000 token context window. Here is how it fills up:

```
CONTEXT WINDOW ANATOMY (per turn):

System prompt (instructions)     ~500 tokens
Assembled context (5 chunks)     ~640 tokens (128 tokens/chunk x 5)
Chat history (per message)       ~100-500 tokens each
Current question                 ~50-100 tokens
----------------------------------------------------
Reserved for generation          remaining tokens
```

Over a long conversation:

```
After 10 messages:   ~1,500 history + ~1,140 fixed = ~2,640 total
After 30 messages:   ~4,500 history + ~1,140 fixed = ~5,640 total
After 100 messages:  ~15,000 history + ~1,140 fixed = ~16,140 total
```

Two things matter more than the hard limit: **cost** (every token costs money) and **quality degradation** (flooding context with old history dilutes important information).

### Truncation vs Summarization

```
| Strategy      | Cost         | Speed    | Early Context | Complexity    | Best For          |
|---------------|--------------|----------|---------------|---------------|-------------------|
| Truncation    | Free (list)  | Instant  | Lost          | 2 lines       | Short sessions    |
| Summarization | Extra API    | 1-2 sec  | Preserved     | ~30 lines     | Long conversations|
```

Today you implement **truncation**. For the Final Project, summarization is a strong upgrade.

### Why Simple Message Count (Not Token Counting)

Token counting adds an API call per turn. For a classroom app with a 200k context window, that overhead is not worth it. Message-count truncation is simple, predictable, and sufficient. A 10-message window is roughly 1,500-5,000 tokens — well within budget.

In [ ]:
# YELLOW scaffold — implement manage_history()
# Implement this in pipeline/context/manager.py


def manage_history(messages: list[dict], max_messages: int = 10) -> list[dict]:
    """Truncate conversation history to fit within context budget.

    Keeps the most recent messages. Old messages are dropped entirely.
    max_messages should be even to preserve complete user/assistant pairs.

    Args:
        messages: Full conversation history (list of role/content dicts).
        max_messages: Maximum number of messages to keep (default: 10,
                      which is 5 complete exchanges).

    Returns:
        Truncated message list (last max_messages messages).

    Notes:
        - Always keep the most recent messages
        - Does NOT include the current user question (that is added after)
        - In production, consider summarization for long conversations
    """
    # Step 1: If messages fit within max_messages, return them all

    # Step 2: Otherwise, return the last max_messages messages (list slice)
    pass

> **Important:** Implement this function in the `.py` file (`pipeline/context/manager.py`), not in this notebook cell. The cell above shows the structure — your implementation must be in the `.py` file for the app to use it. After saving, restart the kernel and run the test cell below.

In [ ]:
# Test your manage_history() implementation
from pipeline.context.manager import manage_history

# Create a fake 15-message conversation
fake_messages = []
for i in range(15):
    role = "user" if i % 2 == 0 else "assistant"
    fake_messages.append({"role": role, "content": f"Message {i+1} from {role}"})

print(f"Total messages: {len(fake_messages)}")
print()

# Truncate to 10
truncated = manage_history(fake_messages, max_messages=10)
print(f"After truncation (max=10): {len(truncated)} messages")
print(f"First kept: {truncated[0]['content']}")
print(f"Last kept:  {truncated[-1]['content']}")
print()

# Under the limit — should return all
short = [{"role": "user", "content": "Hi"}]
result = manage_history(short, max_messages=10)
print(f"Short list (1 message, max=10): {len(result)} messages — unchanged")
print()

# Verify messages 1-5 are dropped, 6-15 remain
assert len(truncated) == 10, f"Expected 10, got {len(truncated)}"
assert truncated[0]["content"] == "Message 6 from assistant", "Wrong first message after truncation"
print("All assertions passed!")

---

## Retrieval Strategies: A Preview

Right now `app/rag.py` uses **naive retrieval** (pure semantic similarity). In Lab 2, you will explore alternative retrieval strategies:

- **HyDE** (Hypothetical Document Embeddings) — generate a hypothetical answer, embed that instead of the question
- **Enriched** — augment the query with additional context before embedding

To switch strategies for Lab 2, you change a single import line in `app/rag.py`:

```python
# Default (naive):
from pipeline.retrieval.naive import naive_retrieve as retrieve

# HyDE (Lab 2 option):
from pipeline.retrieval.hyde import hyde_retrieve as retrieve

# Enriched (Lab 2 option):
from pipeline.retrieval.enriched import enriched_retrieve as retrieve
```

No sidebar toggle, no runtime switching. You pick a strategy, evaluate it, and justify your choice. That is Lab 2.

In [ ]:
# Preview: Current naive retrieval baseline
query = "What is the vacation policy?"

print("NAIVE retrieval (current strategy):")
naive_results = naive_retrieve(query, top_k=3)
for r in naive_results:
    print(f"  [{r['score']:.3f}] {r['metadata'].get('source', 'unknown')}")

print()
print("In Lab 2, you will compare this baseline against HyDE or enriched retrieval.")
print("The question: can a smarter retrieval strategy improve these scores?")

---

## Integration: End-to-End Verification

Let's verify the full pipeline works together: query rewriting + context assembly + history management.

In [ ]:
# Full pipeline test: ask a question, get a response with sources
from app.rag import get_response

# First question — standalone
response = get_response("What is the vacation policy at Northbrook?", messages=[])
print("Question: What is the vacation policy at Northbrook?")
print(f"Answer: {response.answer[:200]}...")
print(f"Sources: {len(response.sources)} chunks")
print(f"Rewritten query: {response.rewritten_query}")
print()

# Verify context assembly is working
assembled = assemble_context(response.sources)
print("Context assembly output (first 300 chars):")
print(assembled[:300])

In [ ]:
# Follow-up test: verify query rewriting improves retrieval scores
from app.rag import get_response

# Simulate conversation history
history = [
    {"role": "user", "content": "What is the vacation policy at Northbrook?"},
    {"role": "assistant", "content": "Northbrook provides paid time off based on employee tenure..."}
]

# Follow-up question
follow_up_response = get_response("How many days do I get?", messages=history)

print("Follow-up: 'How many days do I get?'")
print(f"Rewritten to: {follow_up_response.rewritten_query}")
print()
print(f"Answer: {follow_up_response.answer[:200]}...")
print()
print("Source scores (should be 0.7+ if rewriting works):")
for s in follow_up_response.sources[:3]:
    print(f"  [{s['score']:.3f}] {s['metadata'].get('source', 'unknown')}")

---

## What You Built Today

- **Query Rewriting** (`contextualize_query()`) — follow-up questions are rewritten as standalone queries before retrieval
- **Context Assembly** (`assemble_context()`) — retrieved chunks are grouped by document, sorted by index, gap-marked
- **History Management** (`manage_history()`) — conversation history is truncated to prevent context overflow

### Verification Checklist

- [ ] Follow-up questions retrieve well (scores 0.7+ instead of 0.4)
- [ ] No crashes after 10+ messages
- [ ] Sources display in expander below each response

### Questions to Leave With

> **On Trust:** "You built an app that answers questions from your corpus. What if a user tries to trick it?"

> **On Safety:** "What is the worst thing a user could make your app do?"

---

### Next Session: 3.1 — Prompt Injection & Defensive Design

- What happens when someone tries to break your app?
- "Ignore all previous instructions" — prompt injection attacks
- Defensive strategies and grounding with source material

**Lab 1 Reminder:** Due end of Session 3.2. Your GUI customizations are untouched by today's backend work — different files entirely.